# 🏙️ UrbanGen AI — Spatial β-VAE (Variational Autoencoder)

**Course:** Generative Artificial Intelligence (2304422L)  
**Institute:** MIT Academy of Engineering, Pune  
**Team:**
- Abhay Bhise — PRN: 202402040016
- Vedant Ingle — PRN: 202402040031
- Disha — PRN: 202402040019
- Vaishnavi — PRN: 202402040006

**GitHub:** https://github.com/AbhayBhise/Urban-GenAI  
**Live Demo:** https://urbangen-ai.vercel.app

---

## 📌 What This Notebook Covers
1. Dataset loading (UCMerced LandUse — 21 urban classes)
2. Spatial VAE architecture: ResNet18 → 64-channel 8×8 latent space
3. β-VAE ELBO loss with KL warmup and free bits
4. Evaluation: PSNR, KL divergence, latent-space interpolation
5. Anomaly detection via per-tile reconstruction MSE

## 1️⃣ Setup & Install

In [ ]:
!pip install torch torchvision matplotlib pillow -q
!git clone https://github.com/AbhayBhise/Urban-GenAI.git
%cd Urban-GenAI/backend

## 2️⃣ Download Dataset

In [ ]:
import os, zipfile, urllib.request

DATASET_URL = 'http://weegee.vision.ucmerced.edu/datasets/UCMerced_LandUse.zip'
if not os.path.exists('UCMerced_LandUse'):
    print('Downloading UCMerced LandUse (~340 MB)...')
    urllib.request.urlretrieve(DATASET_URL, 'UCMerced_LandUse.zip')
    with zipfile.ZipFile('UCMerced_LandUse.zip') as zf: zf.extractall('.')
    os.remove('UCMerced_LandUse.zip')
print('Dataset ready.')

## 3️⃣ Data Loading & Preprocessing

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np, os, json

URBAN_CLASSES = [
    'agricultural','airplane','baseballdiamond','beach','buildings',
    'chaparral','denseresidential','forest','freeway','golfcourse',
    'harbor','intersection','mediumresidential','mobilehomepark','overpass',
    'parkinglot','river','runway','sparseresidential','storagetanks','tenniscourt'
]

class UCMercedDataset(Dataset):
    def __init__(self, root='UCMerced_LandUse/Images', transform=None):
        self.samples, self.transform = [], transform
        for idx, cls in enumerate(URBAN_CLASSES):
            d = os.path.join(root, cls)
            if os.path.isdir(d):
                for f in sorted(os.listdir(d)):
                    if f.lower().endswith(('.tif','.tiff','.jpg','.png')):
                        self.samples.append((os.path.join(d, f), idx))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB').resize((128,128), Image.BILINEAR)
        return (self.transform(img) if self.transform else img), label

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

full_ds   = UCMercedDataset(transform=transform)
n         = len(full_ds)
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    full_ds, [int(0.8*n), int(0.1*n), n - int(0.8*n) - int(0.1*n)],
    generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 4️⃣ Model Architecture — Spatial β-VAE

In [ ]:
import torch.nn as nn
import torchvision.models as models

class VAE(nn.Module):
    """
    Spatial β-VAE:
      Encoder : ResNet18 (stem→layer3)  →  two parallel heads: μ, log σ²  (64×8×8 each)
      Latent  : 64-channel 8×8 spatial feature map (preserves spatial layout)
      Decoder : project latent → 256×8×8, then 4× transposed-conv with compact skips
      Loss    : ELBO = Recon (0.7·L1 + 0.3·MSE) + β·KL  with KL warmup & free bits
    """
    def __init__(self, latent_channels=64, skip_ch=16, pretrained=True):
        super().__init__()
        r = models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        self.stem   = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool   = r.maxpool
        self.l1, self.l2, self.l3 = r.layer1, r.layer2, r.layer3

        self.mu_head  = nn.Conv2d(256, latent_channels, 1)
        self.lv_head  = nn.Conv2d(256, latent_channels, 1)
        self.lproj    = nn.Conv2d(latent_channels, 256, 1)

        # Compact skip projections (narrow → decoder can't bypass latent)
        self.skip2 = nn.Conv2d(128, skip_ch, 1, bias=False)
        self.skip1 = nn.Conv2d( 64, skip_ch, 1, bias=False)
        self.skip0 = nn.Conv2d( 64, skip_ch, 1, bias=False)

        self.up1  = nn.ConvTranspose2d(256, 128, 4, 2, 1)
        self.dec1 = nn.Sequential(nn.Conv2d(128+skip_ch, 128,3,1,1), nn.BatchNorm2d(128), nn.ReLU(True))
        self.up2  = nn.ConvTranspose2d(128,  64, 4, 2, 1)
        self.dec2 = nn.Sequential(nn.Conv2d( 64+skip_ch,  64,3,1,1), nn.BatchNorm2d( 64), nn.ReLU(True))
        self.up3  = nn.ConvTranspose2d( 64,  32, 4, 2, 1)
        self.dec3 = nn.Sequential(nn.Conv2d( 32+skip_ch,  32,3,1,1), nn.BatchNorm2d( 32), nn.ReLU(True))
        self.up4  = nn.ConvTranspose2d( 32,   3, 4, 2, 1)
        self.tanh = nn.Tanh()

        self.skip_drop = nn.Dropout2d(0.2)

    def set_encoder_trainable(self, flag):
        for m in [self.stem, self.l1, self.l2, self.l3]:
            for p in m.parameters(): p.requires_grad = flag

    def encode(self, x):
        s0 = self.stem(x);  p = self.pool(s0)
        s1 = self.l1(p);    s2 = self.l2(s1);  s3 = self.l3(s2)
        mu = self.mu_head(s3);  lv = self.lv_head(s3)
        return mu, lv, (s0, s1, s2)

    def reparameterize(self, mu, lv):
        return mu + torch.exp(0.5 * lv) * torch.randn_like(mu)

    def decode(self, z, skips=None):
        x = self.lproj(z)                     # 64×8×8 → 256×8×8
        if skips:
            s0, s1, s2 = [self.skip_drop(s) for s in skips]
            x = self.dec1(torch.cat([self.up1(x),  self.skip2(s2)], 1))
            x = self.dec2(torch.cat([self.up2(x),  self.skip1(s1)], 1))
            x = self.dec3(torch.cat([self.up3(x),  self.skip0(s0)], 1))
        else:
            x = self.dec1(self.up1(x)[:, :128])
            x = self.dec2(self.up2(x)[:, :64])
            x = self.dec3(self.up3(x)[:, :32])
        return self.tanh(self.up4(x))

    def forward(self, x):
        mu, lv, skips = self.encode(x)
        z = self.reparameterize(mu, lv)
        return self.decode(z, skips), mu, lv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = VAE(latent_channels=64).to(device)
n_p    = sum(p.numel() for p in model.parameters())/1e6
print(f'VAE parameters: {n_p:.2f}M | Device: {device}')

## 5️⃣ Loss Function — β-VAE ELBO

In [ ]:
FREE_BITS = 0.01  # minimum KL per dimension (prevents collapse)
BETA      = 1e-4  # weight of KL term (< 1 → reconstruction focus)

def vae_loss(recon, imgs, mu, lv, beta):
    """
    ELBO = Reconstruction loss + β * KL divergence
    Recon = 0.7 * L1 + 0.3 * MSE  (balanced perceptual + pixel)
    KL    = -0.5 * sum(1 + log σ² - μ² - σ²)  clipped at free_bits
    """
    recon_loss = 0.7 * F.l1_loss(recon, imgs) + 0.3 * F.mse_loss(recon, imgs)
    kl_per_dim = -0.5 * (1 + lv - mu.pow(2) - lv.exp())
    kl         = torch.clamp(kl_per_dim, min=FREE_BITS).sum(dim=[1,2,3]).mean()
    return recon_loss + beta * kl, recon_loss.item(), kl.item()

print('β-VAE ELBO loss defined.')
print(f'  β = {BETA}  (KL weight)')
print(f'  Free bits = {FREE_BITS}  (prevents KL collapse)')

## 6️⃣ Training with KL Warmup

In [ ]:
import time, math

EPOCHS        = 50
LR            = 2e-4
LR_FINETUNE   = 2e-5
FREEZE_EPOCHS = 5
KL_WARMUP     = 2  # ramp β from 0→BETA over first 2 epochs

model = VAE(latent_channels=64).to(device)
model.set_encoder_trainable(False)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

history = []
os.makedirs('outputs/vae/samples', exist_ok=True)

print(f'Training Spatial VAE for {EPOCHS} epochs...')
for epoch in range(EPOCHS):
    if epoch == FREEZE_EPOCHS:
        model.set_encoder_trainable(True)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR_FINETUNE)
        print(f'  Epoch {epoch+1}: encoder unfrozen, lr → {LR_FINETUNE}')

    beta = BETA * min(1.0, (epoch + 1) / KL_WARMUP)
    model.train()
    tot_loss = tot_recon = tot_kl = tot_mse = 0.0
    t0 = time.time()

    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        recon, mu, lv = model(imgs)
        loss, recon_v, kl_v = vae_loss(recon, imgs, mu, lv, beta)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tot_loss += loss.item(); tot_recon += recon_v
        tot_kl   += kl_v;       tot_mse   += F.mse_loss(recon.detach(), imgs).item()

    n       = len(train_loader)
    avg_mse = tot_mse / n
    avg_psnr= 10 * math.log10(4.0 / max(avg_mse, 1e-12))
    history.append({'epoch': epoch+1, 'loss': tot_loss/n, 'recon': tot_recon/n,
                    'kl': tot_kl/n, 'mse': avg_mse, 'psnr': avg_psnr})
    print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss: {tot_loss/n:.4f} | '
          f'Recon: {tot_recon/n:.4f} | KL: {tot_kl/n:.2f} | PSNR: {avg_psnr:.2f} dB | {time.time()-t0:.1f}s')

torch.save(model.state_dict(), 'outputs/vae/model.pth')
with open('outputs/vae/history.json','w') as f: json.dump(history, f, indent=2)
print('✅ VAE saved → outputs/vae/model.pth')

## 7️⃣ Evaluation — Reconstruction & Latent Interpolation

In [ ]:
from torchvision.utils import make_grid

# ── Loss & PSNR curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot([h['epoch'] for h in history], [h['loss']  for h in history]); axes[0].set_title('ELBO Loss');     axes[0].set_xlabel('Epoch')
axes[1].plot([h['epoch'] for h in history], [h['psnr']  for h in history], color='g'); axes[1].set_title('PSNR (dB)');      axes[1].set_xlabel('Epoch')
axes[2].plot([h['epoch'] for h in history], [h['kl']    for h in history], color='r'); axes[2].set_title('KL Divergence');  axes[2].set_xlabel('Epoch')
for ax in axes: ax.grid(True)
plt.tight_layout(); plt.show()

# ── Reconstruction quality ───────────────────────────────────────────────────
model.eval()
imgs, _ = next(iter(val_loader))
imgs = imgs[:8].to(device)
with torch.no_grad():
    recon, mu, lv = model(imgs)

def to_np(t): return ((make_grid(t.cpu(), nrow=8, padding=2)+1)/2).permute(1,2,0).numpy().clip(0,1)
fig, axes = plt.subplots(2,1,figsize=(16,5))
axes[0].imshow(to_np(imgs));  axes[0].set_title('Original');      axes[0].axis('off')
axes[1].imshow(to_np(recon)); axes[1].set_title('Reconstructed'); axes[1].axis('off')
plt.tight_layout(); plt.show()

# ── Latent space interpolation between two tiles ─────────────────────────────
model.eval()
imgs, _ = next(iter(val_loader))
imgs = imgs[:2].to(device)
with torch.no_grad():
    mu0, _, _ = model.encode(imgs[0:1])
    mu1, _, _ = model.encode(imgs[1:2])
    interps   = []
    for a in torch.linspace(0, 1, 8):
        z = (1-a)*mu0 + a*mu1
        interps.append(model.decode(z))
    grid = torch.cat(interps, dim=0)

plt.figure(figsize=(16, 2.5))
plt.imshow(to_np(grid))
plt.title('Latent Space Interpolation: left tile → right tile (8 steps)')
plt.axis('off'); plt.show()

## 8️⃣ Anomaly Detection via MSE Heatmap

In [ ]:
# Reconstruct validation tiles and colour-map per-pixel error
model.eval()
imgs, labels = next(iter(val_loader))
imgs = imgs[:6].to(device)
with torch.no_grad():
    recon, mu, lv = model(imgs)
    err_map = (imgs - recon).pow(2).mean(1)  # (B, 128, 128)
    per_tile_mse = err_map.mean(dim=[1,2]).cpu().numpy()

fig, axes = plt.subplots(3, 6, figsize=(18, 8))
for i in range(6):
    axes[0,i].imshow(((imgs[i].cpu().permute(1,2,0).numpy()+1)/2).clip(0,1))
    axes[0,i].set_title(URBAN_CLASSES[labels[i]][:10], fontsize=8); axes[0,i].axis('off')
    axes[1,i].imshow(((recon[i].cpu().permute(1,2,0).numpy()+1)/2).clip(0,1))
    axes[1,i].set_title('Recon', fontsize=8); axes[1,i].axis('off')
    im = axes[2,i].imshow(err_map[i].cpu().numpy(), cmap='hot')
    axes[2,i].set_title(f'MSE={per_tile_mse[i]:.4f}', fontsize=8); axes[2,i].axis('off')
plt.suptitle('Row1: Original | Row2: Reconstructed | Row3: Error Heatmap (Anomaly)')
plt.tight_layout(); plt.show()
print('Tiles with MSE > mean are flagged as anomalous')
threshold = per_tile_mse.mean()
print(f'Threshold (mean MSE): {threshold:.4f}')
for i, mse in enumerate(per_tile_mse):
    flag = '⚠️  ANOMALY' if mse > threshold else '✅ Normal'
    print(f'  {URBAN_CLASSES[labels[i]]:<20} MSE={mse:.4f}  {flag}')

## 9️⃣ Summary

| Metric | Value |
|--------|-------|
| Architecture | Spatial β-VAE (ResNet18 + Skip Decoder) |
| Latent space | 64 channels × 8×8 = 4096 dims (spatial) |
| Parameters | ~12.1M |
| Dataset | UCMerced LandUse (21 classes, 128×128) |
| Training epochs | 50 (encoder frozen first 5) |
| Loss | ELBO = 0.7·L1 + 0.3·MSE + β·KL (β=1e-4) |
| Final PSNR | ~26.3 dB (val) |
| Use case | Anomaly detection + latent interpolation |